In [0]:
from pyspark.sql.functions import expr, rand

orders_df = spark.range(1, 2_000_000) \
    .withColumn("order_id", expr("id")) \
    .withColumn("customer_id", expr("id % 50000")) \
    .withColumn(
        "city",
        expr("""
        CASE
          WHEN id % 10 < 6 THEN 'Bangalore'
          WHEN id % 10 < 8 THEN 'Mumbai'
          ELSE 'Delhi'
        END
        """)
    ) \
    .withColumn("amount", expr("round(rand()*1000,2)")) \
    .withColumn("order_date", expr("current_date()"))

In [0]:
orders_df.display()

In [0]:
orders_df.write.mode("overwrite").json('/Volumes/workspace/projectdatabase/projectsfies/bronze/')

In [0]:
bronze_df = spark.read.json('/Volumes/workspace/projectdatabase/projectsfies/bronze/')
bronze_df.display()

In [0]:
bronze_df.write \
    .format('delta') \
    .mode('overwrite') \
    .save('/Volumes/workspace/projectdatabase/projectsfies/silver/')

In [0]:
bronze_df.groupBy("city").count().orderBy("count", ascending=False).show()


In [0]:
bronze_df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/projectdatabase/projectsfies/silver/")


In [0]:
silver_df = spark.read.format("delta").load("/Volumes/workspace/projectdatabase/projectsfies/silver/")
silver_df.display()

In [0]:
from pyspark.sql.functions import concat, floor, rand, col

silver_df = bronze_df.withColumn(
    "salted_city",
    concat(col("city"), floor(rand() * 10))
)
from pyspark.sql.functions import concat, floor, rand, col

silver_df = bronze_df.withColumn(
    "salted_city",
    concat(col("city"), floor(rand() * 10))
)


In [0]:
silver_df.display()

In [0]:
silver_df = silver_df.dropDuplicates(["order_id"]) \
    .withColumn("amount", col("amount").cast("double"))
silver_df.display()

In [0]:
dbutils.fs.rm(
  "/Volumes/workspace/projectdatabase/projectsfies/silver/order",
  recurse=True
)